# Lab 16 - Cloud AI Benchmark on Kaggle Notebook

This notebook is the free fallback path for Lab 16 when AWS/Oracle account creation is blocked.

It runs the required CPU LightGBM benchmark on the Credit Card Fraud Detection dataset and writes `benchmark_result.json`.


## Step 1 - Verify Dataset Path

On Kaggle, open **Add Data** and attach the dataset **Credit Card Fraud Detection** (`mlg-ulb/creditcardfraud`).

The expected file path is:

```text
/kaggle/input/creditcardfraud/creditcard.csv
```


In [ ]:
from pathlib import Path

matches = sorted(Path('/kaggle/input').glob('**/creditcard.csv'))
if not matches:
    raise FileNotFoundError('Could not find creditcard.csv under /kaggle/input. Please add the Credit Card Fraud Detection dataset.')

DATA_PATH = matches[0]
OUTPUT_PATH = Path('/kaggle/working/benchmark_result.json')

print('Dataset exists:', DATA_PATH.exists())
print('Dataset path:', DATA_PATH)
print('Output path:', OUTPUT_PATH)


## Step 2 - Install/Import Dependencies

Kaggle usually already has these packages. The install line is kept quiet and safe if LightGBM is already present.


In [ ]:
import sys
import subprocess

subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lightgbm', '-q'])

import json
import os
import platform
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

print('Python:', platform.python_version())
print('Platform:', platform.platform())


## Step 3 - Run LightGBM Benchmark

This cell measures:

- Data load time
- Training time
- Best iteration
- AUC-ROC, Accuracy, F1, Precision, Recall
- Single-row inference latency
- 1000-row inference throughput


In [ ]:
def seconds_since(start_time):
    return round(time.perf_counter() - start_time, 6)


def milliseconds_since(start_time):
    return round((time.perf_counter() - start_time) * 1000, 6)


def print_metric(name, value):
    print(f'{name:<40} {value}')


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found: {DATA_PATH}. Use Kaggle Add Data and attach mlg-ulb/creditcardfraud.'
    )

print('Lab 16 LightGBM Credit Card Fraud Benchmark')
print(f'Dataset: {DATA_PATH}')
print()

load_started = time.perf_counter()
data = pd.read_csv(DATA_PATH)
load_time_seconds = seconds_since(load_started)

expected_columns = {'Class', 'Amount', 'Time'}
missing_columns = sorted(expected_columns - set(data.columns))
if missing_columns:
    raise ValueError(f'Dataset is missing expected columns: {missing_columns}')

y = data['Class'].astype(int)
x = data.drop(columns=['Class'])

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

positive_count = int(y_train.sum())
negative_count = int(len(y_train) - positive_count)
scale_pos_weight = negative_count / positive_count

model = lgb.LGBMClassifier(
    objective='binary',
    boosting_type='gbdt',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
)

training_started = time.perf_counter()
model.fit(
    x_train,
    y_train,
    eval_set=[(x_test, y_test)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(30, verbose=False)],
)
training_time_seconds = seconds_since(training_started)

prediction_started = time.perf_counter()
y_probability = model.predict_proba(x_test)[:, 1]
batch_prediction_time_seconds = seconds_since(prediction_started)
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probability)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-12)
best_index = int(np.nanargmax(f1_scores[:-1]))
best_threshold = float(thresholds[best_index])
y_predicted = (y_probability >= best_threshold).astype(int)
metrics_threshold = round(best_threshold, 6)

one_row = x_test.iloc[[0]]
latency_started = time.perf_counter()
model.predict_proba(one_row)
single_row_latency_ms = milliseconds_since(latency_started)

thousand_rows = x_test.iloc[:1000]
throughput_started = time.perf_counter()
model.predict_proba(thousand_rows)
thousand_row_time_seconds = seconds_since(throughput_started)
rows_per_second = round(len(thousand_rows) / thousand_row_time_seconds, 3)

metrics = {
    'decision_threshold': metrics_threshold,
    'environment': 'Kaggle Notebook CPU',
    'dataset_path': str(DATA_PATH),
    'rows': int(len(data)),
    'features': int(x.shape[1]),
    'fraud_rows': int(y.sum()),
    'fraud_rate': round(float(y.mean()), 8),
    'train_rows': int(len(x_train)),
    'test_rows': int(len(x_test)),
    'load_time_seconds': load_time_seconds,
    'training_time_seconds': training_time_seconds,
    'best_iteration': int(model.best_iteration_ or model.n_estimators),
    'auc_roc': round(float(roc_auc_score(y_test, y_probability)), 6),
    'accuracy': round(float(accuracy_score(y_test, y_predicted)), 6),
    'f1_score': round(float(f1_score(y_test, y_predicted, zero_division=0)), 6),
    'precision': round(float(precision_score(y_test, y_predicted, zero_division=0)), 6),
    'recall': round(float(recall_score(y_test, y_predicted, zero_division=0)), 6),
    'single_row_inference_latency_ms': single_row_latency_ms,
    'thousand_row_inference_time_seconds': thousand_row_time_seconds,
    'inference_throughput_rows_per_second': rows_per_second,
    'batch_prediction_time_seconds': batch_prediction_time_seconds,
    'python_version': platform.python_version(),
    'platform': platform.platform(),
}

OUTPUT_PATH.write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')

print('Benchmark results')
print('-' * 70)
print_metric('Rows', metrics['rows'])
print_metric('Fraud rows', metrics['fraud_rows'])
print_metric('Fraud rate', metrics['fraud_rate'])
print_metric('Load data time (seconds)', metrics['load_time_seconds'])
print_metric('Training time (seconds)', metrics['training_time_seconds'])
print_metric('Best iteration', metrics['best_iteration'])
print_metric('AUC-ROC', metrics['auc_roc'])
print_metric('Accuracy', metrics['accuracy'])
print_metric('F1-score', metrics['f1_score'])
print_metric('Precision', metrics['precision'])
print_metric('Recall', metrics['recall'])
print_metric('Inference latency, 1 row (ms)', metrics['single_row_inference_latency_ms'])
print_metric('Inference throughput, 1000 rows/s', metrics['inference_throughput_rows_per_second'])
print('-' * 70)
print(f'Saved JSON result to: {OUTPUT_PATH}')


## Step 4 - Show JSON Result

Download `benchmark_result.json` from the Kaggle **Output** panel after running the notebook.


In [ ]:
print(OUTPUT_PATH.read_text())


## Step 5 - Resource Evidence

Run this cell and screenshot the output to show CPU/RAM/runtime evidence from the free cloud notebook environment.


In [ ]:
import multiprocessing

print('CPU count:', multiprocessing.cpu_count())
print('Python executable:', sys.executable)
print('Working directory files:')
print(os.listdir('/kaggle/working'))

try:
    with open('/proc/meminfo', 'r', encoding='utf-8') as file:
        for line in file.readlines()[:5]:
            print(line.strip())
except FileNotFoundError:
    print('/proc/meminfo is not available in this environment')


## Short Report Template

Use the measured values above to fill this report.

```text
Because AWS account verification and Oracle Cloud sign-up were blocked, I completed the CPU benchmark on Kaggle Notebook, a free managed cloud notebook environment. I used the Credit Card Fraud Detection dataset with 284,807 transactions and trained a LightGBM binary classifier. The benchmark measured data loading time, training time, AUC-ROC, Accuracy, F1-score, Precision, Recall, single-row inference latency, and 1000-row inference throughput. The model achieved AUC-ROC = <AUC>, Accuracy = <ACC>, F1 = <F1>, Precision = <PRECISION>, and Recall = <RECALL>. Training took <TRAINING_TIME> seconds. Single-row inference latency was <LATENCY> ms and 1000-row throughput was <THROUGHPUT> rows/second. The result was saved to benchmark_result.json. This path used CPU only, no GPU, and no paid cloud resources.
```
